# Thêm Thư Viện

In [1]:
import pyodbc
import pandas as pd
import numpy as np

# Tạo kết nối

In [2]:
conn_libol = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=192.168.150.6;'  # Địa chỉ IP của SQL Server
    'DATABASE=libol;'         # Tên cơ sở dữ liệu
    'UID=itc;'                # Tên đăng nhập
    'PWD=spkt@2025;'
)
conn_dwh_library = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=192.168.150.6;' # Địa chỉ IP của SQL Server
    'DATABASE=Library_DWH;' # Tên cơ sở dữ liệu
    'UID=itc;'              # Tên đăng nhập
    'PWD=spkt@2025;'
)

## Đọc data từ SQL Server

In [3]:
# Đọc dữ liệu từ bảng Lop trong CSDL libol
query_Nhombandoc = "SELECT Nhom_ID, dbo.DecodeUTF8String(Ten_nhom) AS Ten_nhom FROM Nhom_ban_doc"
df_nhombandoc = pd.read_sql(query_Nhombandoc, conn_libol)
print(df_nhombandoc)

    Nhom_ID                  Ten_nhom
0         5                          
1         6         .Cán bộ công chức
2         9          MƯỢN & ĐỌC - SKV
3        10  Tốt nghiệp_Cộng Tác viên
4        12               Đọc tại chỗ
5        14                   MƯỢN GT
6        15             MƯỢN GT & SKV
7        16    Chưa tham gia khóa học
8        17    Con CB & CTV (Mượn GT)
9        18          HỌC VIÊN CAO HỌC
10       19    Khoa ĐT chất lượng cao
11       20         NHÓM NGOÀI TRƯỜNG
12       21       GIÁO TRÌNH QUÉT LỘN
13       22    NHÓM LÃNH ĐẠO, QUẢN LÝ
14       23         Nhóm ngoài trường
15       24      Nhóm ký công nợ (TN)
16       25           Nghiên cứu sinh


C:\Users\admin\AppData\Local\Temp\ipykernel_19648\1031425255.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_nhombandoc = pd.read_sql(query_Nhombandoc, conn_libol)


## Xử lý data

In [4]:
df_nhombandoc = df_nhombandoc.drop_duplicates(subset='Ten_nhom').reset_index(drop=True) # xóa những hàng có Ten_nhom bị trùng nhau

new_row = pd.DataFrame({'Nhom_ID': [0], 'Ten_nhom': ['(Không xác định)']}) # Tạo hàng dữ liệu giả lập cho nhóm không xác định
df_nhombandoc = pd.concat([df_nhombandoc, new_row], ignore_index=True) # Thêm vào dataframe

for j, row in df_nhombandoc.iterrows(): # quét qua dữ liệu từng hàng của cột Ten_nhom
    ten_nhom = row["Ten_nhom"]
    if pd.isna(ten_nhom) or ten_nhom == "":  # Kiểm tra NaN or None
        df_nhombandoc.at[j, 'Ten_nhom'] = "(Không xác định)"
    else:
        ten_nhom = ten_nhom[0].upper() + ten_nhom[1:].lower() # Chỉnh sửa chữ hoa chữ thường nếu chuỗi không rỗng
        df_nhombandoc.at[j, 'Ten_nhom'] = ten_nhom

df_nhombandoc = df_nhombandoc.sort_values(by='Nhom_ID', ascending=True).reset_index(drop=True) # sắp xếp lại cho dễ nhìn
print(df_nhombandoc)

    Nhom_ID                  Ten_nhom
0         0          (không xác định)
1         5          (Không xác định)
2         6         .cán bộ công chức
3         9          Mượn & đọc - skv
4        10  Tốt nghiệp_cộng tác viên
5        12               Đọc tại chỗ
6        14                   Mượn gt
7        15             Mượn gt & skv
8        16    Chưa tham gia khóa học
9        17    Con cb & ctv (mượn gt)
10       18          Học viên cao học
11       19    Khoa đt chất lượng cao
12       20         Nhóm ngoài trường
13       21       Giáo trình quét lộn
14       22    Nhóm lãnh đạo, quản lý
15       23         Nhóm ngoài trường
16       24      Nhóm ký công nợ (tn)
17       25           Nghiên cứu sinh


## Load data

### [Nếu cần] Clear bảng

In [5]:
cursor = conn_dwh_library.cursor()
truncate_query = "DELETE FROM olap.DIM_Nhom_ban_doc"
cursor.execute(truncate_query)
conn_dwh_library.commit()
cursor.close()

### Load data vào bảng Dim

In [7]:
cursor_dwh = conn_dwh_library.cursor()
insert_query = """
                INSERT INTO olap.DIM_Nhom_ban_doc (ID_nhom_ban_doc, Nhom_ban_doc) 
                VALUES (?, ?)
                """
for index, row in df_nhombandoc.iterrows():
    values = (row['Nhom_ID'], 
              row['Ten_nhom'])
    cursor_dwh.execute(insert_query, values)
conn_dwh_library.commit()